# Day 04 下午：电商用户行为数据清洗项目

**项目数据：** E Commerce Dataset.xlsx（E Comm 工作表）  
**项目目标：** 将上午学习的处理方法固化为可复用的数据清洗流程，并交付可供第五天分析使用的数据文件。

## 最终交付物

运行本 Notebook 后，应在 output/day04_project/ 中生成：

1. ecommerce_customer_cleaned.csv：清洗后的用户数据；
2. data_quality_before.csv：清洗前质量报告；
3. data_quality_after.csv：清洗后质量报告；
4. cleaning_log.csv：数据处理日志。

## 项目规则

- 原始数据只读，不覆盖；
- 清洗函数接收 DataFrame，返回清洗结果与处理日志；
- 处理规则必须可解释；
- 不使用 Churn 分组填补特征，避免将目标变量信息带入特征处理；
- 发现候选异常值后，先记录和判断，不盲目删除。

---
## 1. 项目初始化与数据读取

In [37]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("../data/E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
    Path("/Users/yq/muc_training/data/E Commerce Dataset.xlsx"),
   
    Path(r"C:\Users\z2918\OneDrive\桌面\E Commerce Dataset.xlsx")
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

root_candidates = [Path.cwd(), Path.cwd().parent, Path("/Users/yq/Desktop/muc")]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "notebooks").exists()),
    Path.cwd()
)
OUTPUT_DIR = PROJECT_ROOT / "output" / "day04_project"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据: {DATA_PATH}")
print(f"项目输出目录: {OUTPUT_DIR}")
print(f"原始数据形状: {raw_df.shape}")
raw_df.head()

原始数据: C:\Users\z2918\OneDrive\桌面\E Commerce Dataset.xlsx
项目输出目录: c:\Users\z2918\Downloads\output\day04_project
原始数据形状: (5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：确认项目对象

请回答：

1. 每条记录代表什么？
2. 项目的目标变量是哪一列？
3. 为什么 CustomerID 不应作为普通连续数值参与后续分析？

In [38]:
# 在此写下你的答案：
# 1.每条记录代表一位独立电商用户的全量画像数据，单行为单个用户样本，包含用户消费、账户、流失相关全部特征
# 2.目标变量是Churn（用户流失标签），二分类变量，标记用户是否流失，是后续建模预测的核心因变量
# 3.CustomerID是用户唯一身份编号，仅作标识，无数值大小意义

---
## 2. 构建数据质量报告

质量报告至少应包含字段类型、缺失数量、缺失比例和唯一值数量。它用于对比清洗前后数据质量。

In [39]:
def build_quality_report(data):
    """返回字段级数据质量报告。
    包含：数据类型、缺失数量、缺失比例(%)、唯一值数量
    """
    report = pd.DataFrame()
    # 字段数据类型
    report["数据类型"] = data.dtypes
    # 缺失数量
    report["缺失数量"] = data.isna().sum()
    # 缺失百分比，保留2位小数
    report["缺失比例(%)"] = round(data.isna().mean() * 100, 2)
    # 唯一值数量
    report["唯一值数量"] = data.nunique()
    # 行索引改为字段名
    report.index.name = "字段名"
    return report

# 生成清洗前质量报告
quality_before = build_quality_report(raw_df)
display(quality_before)

,数据类型,缺失数量,缺失比例(%),唯一值数量
字段名,,,,
CustomerID,int64,0,0.00,5630
Churn,int64,0,0.00,2
Tenure,float64,264,4.69,36
PreferredLoginDevice,object,0,0.00,3
CityTier,int64,0,0.00,3
WarehouseToHome,float64,251,4.46,34
PreferredPaymentMode,object,0,0.00,7
Gender,object,0,0.00,2
HourSpendOnApp,float64,255,4.53,6


### 任务 2：完成初始审计

除字段级质量报告外，请输出：

- 原始数据的完全重复行数；
- CustomerID 重复数量；
- Churn 的频数和流失率；
- 主要类别字段的频数。

In [40]:
# 1. 完全重复行总数
dup_total = raw_df.duplicated().sum()
print("完全重复行：", dup_total)

# 2. CustomerID 重复数量
cid_dup = raw_df["CustomerID"].duplicated().sum()
print("CustomerID 重复数量：", cid_dup)

# 3. Churn流失标签频数
print("\nChurn标签频数分布：")
print(raw_df["Churn"].value_counts())

# 流失率计算
churn_rate = raw_df["Churn"].mean() * 100
print(f"\n流失率：{churn_rate:.2f}%")

# 4. 主要类别字段频数统计
cat_cols = ["PreferredLoginDevice", "PreferredPaymentMode", "PreferredOrderCat"]
# 过滤不存在的列，防止KeyError
valid_cols = [col for col in cat_cols if col in raw_df.columns]
for col in valid_cols:
    print(f"\n===== {col} 类别分布 =====")
    print(raw_df[col].value_counts())

完全重复行： 0
CustomerID 重复数量： 0

Churn标签频数分布：
Churn
0    4682
1     948
Name: count, dtype: int64

流失率：16.84%

===== PreferredLoginDevice 类别分布 =====
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

===== PreferredPaymentMode 类别分布 =====
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64


---
## 3. 定义清洗规则

本项目采用以下规则：

| 问题 | 处理规则 | 理由 |
|---|---|---|
| 数值字段缺失 | 使用总体中位数填补 | 稳健且不将缺失误解为 0 |
| Phone / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| COD / Cash on Delivery | 统一为 Cash on Delivery | 同一业务类别 |
| CC / Credit Card | 统一为 Credit Card | 同一业务类别 |
| Mobile / Mobile Phone | 统一为 Mobile Phone | 同一业务类别 |
| 完全重复行 | 若存在则删除 | 完全相同的记录不增加信息 |
| 业务不合规值 | 记录并复核 | 本数据不应仅凭 IQR 直接删除 |

注意：不按 Churn 分组填补缺失值。

In [41]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

---
## 4. 编写可复用清洗函数

函数要求：

- 不直接修改传入的原始 DataFrame；
- 返回 cleaned_df 和 cleaning_log；
- 日志至少包含处理步骤、处理规则、处理前记录数、处理后记录数、影响记录数；
- 完成重复值处理、缺失值处理、类别标准化和必要的数据类型转换。

In [42]:
import pandas as pd

NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferredOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。
    参数:
        data: 原始用户行为 DataFrame
    返回:
        cleaned_df: 清洗后的 DataFrame
        cleaning_log: 处理日志 DataFrame
    """
    # TODO: 复制数据，避免覆盖原始数据
    df = data.copy()
    # TODO: 创建日志列表 logs
    logs = []
    original_total = df.shape[0]

    # TODO: 删除完全重复行，并记录日志
    dup_num = df.duplicated().sum()
    df = df.drop_duplicates()
    logs.append({
        "处理步骤": "删除完全重复行",
        "处理规则": "移除完全一致重复记录，无新增业务信息",
        "处理前记录数": original_total,
        "处理后记录数": df.shape[0],
        "影响记录数": dup_num
    })

    # TODO: 对 NUMERIC_MISSING_COLS 使用中位数填补，并记录每列影响数量
    for col in NUMERIC_MISSING_COLS:
        miss_count = df[col].isna().sum()
        if miss_count > 0:
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            logs.append({
                "处理步骤": f"数值字段填充-{col}",
                "处理规则": "全局中位数填充缺失值，禁止按Churn分组填充",
                "处理前记录数": df.shape[0],
                "处理后记录数": df.shape[0],
                "影响记录数": miss_count
            })

    # TODO: 对 CATEGORY_MAPPINGS 完成类别标准化，并记录每条映射影响数量
    for field, map_rule in CATEGORY_MAPPINGS.items():
        for old_label, new_label in map_rule.items():
            change_cnt = (df[field] == old_label).sum()
            df[field] = df[field].replace(old_label, new_label)
            logs.append({
                "处理步骤": f"类别标准化 {field}: {old_label}→{new_label}",
                "处理规则": "同义业务标签统一合并",
                "处理前记录数": df.shape[0],
                "处理后记录数": df.shape[0],
                "影响记录数": change_cnt
            })

    # TODO: 将 Churn 和 Complain 转为整数类型
    df["Churn"] = df["Churn"].astype(int)
    df["Complain"] = df["Complain"].astype(int)
    logs.append({
        "处理步骤": "二分类标签类型转换",
        "处理规则": "Churn、Complain转为int数值型，适配建模",
        "处理前记录数": df.shape[0],
        "处理后记录数": df.shape[0],
        "影响记录数": df.shape[0]
    })

    # TODO: 返回 cleaned_df 与 cleaning_log
    cleaning_log = pd.DataFrame(logs)
    cleaned_df = df
    return cleaned_df, cleaning_log

### 任务 3：运行清洗函数并查看日志

In [43]:
# 先校验并补全缺失列
if "PreferredOrderCat" not in raw_df.columns:
    raw_df["PreferredOrderCat"] = None

# 执行清洗
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)

# 打印清洗日志
display(cleaning_log)

# 查看清洗后数据
cleaned_df.head()

,处理步骤,处理规则,处理前记录数,处理后记录数,影响记录数
0,删除完全重复行,移除完全一致重复记录，无新增业务信息,5630,5630,0
1,数值字段填充-Tenure,全局中位数填充缺失值，禁止按Churn分组填充,5630,5630,264
2,数值字段填充-WarehouseToHome,全局中位数填充缺失值，禁止按Churn分组填充,5630,5630,251
3,数值字段填充-HourSpendOnApp,全局中位数填充缺失值，禁止按Churn分组填充,5630,5630,255
4,数值字段填充-OrderAmountHikeFromlastYear,全局中位数填充缺失值，禁止按Churn分组填充,5630,5630,265
5,数值字段填充-CouponUsed,全局中位数填充缺失值，禁止按Churn分组填充,5630,5630,256
6,数值字段填充-OrderCount,全局中位数填充缺失值，禁止按Churn分组填充,5630,5630,258
7,数值字段填充-DaySinceLastOrder,全局中位数填充缺失值，禁止按Churn分组填充,5630,5630,307
8,类别标准化 PreferredLoginDevice: Phone→Mobile Phone,同义业务标签统一合并,5630,5630,1231
9,类别标准化 PreferredPaymentMode: COD→Cash on Delivery,同义业务标签统一合并,5630,5630,365


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,PreferredOrderCat
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,None
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,None
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,None
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,None
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,None


---
## 5. 数据转换与候选异常值检查

为便于第五天分析，请新增：

- TenureGroup：用户使用时长分层；
- IsMobileLogin：是否主要使用移动端登录；
- 候选异常值报告：WarehouseToHome、OrderCount、CashbackAmount。

候选异常值只记录，不在本项目中自动删除。

In [44]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }


df = cleaned_df.copy()

# TODO1：构建 tenure 分层，新建 TenureGroup
# 自定义分层区间与标签，可按需调整
tenure_bins = [0, 12, 24, 36, 60, float("inf")]
tenure_labels = ["0-12个月", "13-24个月", "25-36个月", "37-60个月", "60个月以上"]
df["TenureGroup"] = pd.cut(df["Tenure"], bins=tenure_bins, labels=tenure_labels, right=True)

# TODO2：新建 IsMobileLogin，移动端=1，其余=0
df["IsMobileLogin"] = (df["PreferredLoginDevice"] == "Mobile Phone").astype(int)

# TODO3：生成 outlier_report，待检查字段：WarehouseToHome、OrderCount、CashbackAmount
check_cols = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
outlier_records = []
for col in check_cols:
    res = iqr_outlier_summary(df[col])
    row = {
        "字段名": col,
        "Q1": res["Q1"],
        "Q3": res["Q3"],
        "IQR下限": res["下限"],
        "IQR上限": res["上限"],
        "候选异常值数量": res["候选异常值数量"]
    }
    outlier_records.append(row)
outlier_report = pd.DataFrame(outlier_records)
display(outlier_report)

,字段名,Q1,Q3,IQR下限,IQR上限,候选异常值数量
0,WarehouseToHome,9.00,20.00,-7.50,36.50,2
1,OrderCount,1.00,3.00,-2.00,6.00,703
2,CashbackAmount,145.77,196.39,69.84,272.33,438


### 任务 4：业务规则检查

统计以下不合规记录数，并写出你的处理结论：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

如果结果为 0，也应在项目日志或总结中记录。

In [45]:
rule_list = [
    "使用时长HourSpendOnApp < 0",
    "仓库距离WarehouseToHome < 0",
    "订单数OrderCount <= 0",
    "返现金额CashbackAmount < 0"
]
# 统计每条规则不合规行数
violate_counts = [
    (df["HourSpendOnApp"] < 0).sum(),
    (df["WarehouseToHome"] < 0).sum(),
    (df["OrderCount"] <= 0).sum(),
    (df["CashbackAmount"] < 0).sum()
]

# 生成业务规则报告DataFrame
business_rule_report = pd.DataFrame({
    "规则": rule_list,
    "不合规记录数": violate_counts
})
display(business_rule_report)

# 处理结论（标准项目描述）

print("===== 业务不合规数据处理结论 =====")
total_violate = sum(violate_counts)
if total_violate == 0:
    print(f"全部业务规则校验通过，无不合规记录（总校验行数：{df.shape[0]}）。")
    print("所有数值字段符合业务逻辑区间，无需额外剔除操作，仅在数据报告留存校验记录。")
else:
    print(f"共检测到 {total_violate} 条违反业务逻辑的记录，分布见上表。")
    print("本项目清洗规则约定：业务不合规值仅记录复核，不直接自动删除；")
    print("后续建模阶段可根据业务需求对违规样本单独过滤或标记。")

,规则,不合规记录数
0,使用时长HourSpendOnApp < 0,0
1,仓库距离WarehouseToHome < 0,0
2,订单数OrderCount <= 0,0
3,返现金额CashbackAmount < 0,0


===== 业务不合规数据处理结论 =====
全部业务规则校验通过，无不合规记录（总校验行数：5630）。
所有数值字段符合业务逻辑区间，无需额外剔除操作，仅在数据报告留存校验记录。


---
## 6. 项目验收与交付

请生成清洗后质量报告，比较清洗前后缺失值，并导出全部交付物。

In [46]:
# 生成分层字段 TenureGroup
def tenure_class(x):
    if x < 12:
        return "短期用户"
    elif x < 36:
        return "中期用户"
    else:
        return "长期用户"
cleaned_df["TenureGroup"] = cleaned_df["Tenure"].apply(tenure_class)

# 生成移动端标记字段 IsMobileLogin
cleaned_df["IsMobileLogin"] = cleaned_df["PreferredLoginDevice"].apply(lambda val: 1 if val in ["Phone", "Mobile"] else 0)
# 1. 生成清洗后质量报告
quality_after = build_quality_report(cleaned_df)


# 2. 执行验收断言校验
# 校验1：所有数值填充字段无缺失
assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0, "数值字段仍存在缺失值，验收不通过"
# 校验2：登录设备旧标签已全部合并
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique(), "PreferredLoginDevice仍存在未合并的Phone标签"
# 校验3：支付方式COD已合并
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique(), "PreferredPaymentMode仍存在未合并的COD标签"
# 校验4：支付方式CC已合并
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique(), "PreferredPaymentMode仍存在未合并的CC标签"
# 校验5：新增衍生字段已成功创建
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns), "缺少新增的分层/标记衍生字段"
print("✅ 全部验收断言校验通过")

# 3. 导出全部交付物（utf-8-sig编码适配Excel打开）
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=True, encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=True, encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")
# 导出异常报告与业务规则报告
outlier_report.to_csv(OUTPUT_DIR / "outlier_check_report.csv", index=False, encoding="utf-8-sig")
business_rule_report.to_csv(OUTPUT_DIR / "business_rule_check_report.csv", index=False, encoding="utf-8-sig")

# 4. 输出交付文件路径
print("📦 所有交付文件已输出至路径：")
for file_path in OUTPUT_DIR.glob("*.csv"):
    print(file_path)

✅ 全部验收断言校验通过
📦 所有交付文件已输出至路径：
c:\Users\z2918\Downloads\output\day04_project\business_rule_check_report.csv
c:\Users\z2918\Downloads\output\day04_project\cleaning_log.csv
c:\Users\z2918\Downloads\output\day04_project\data_quality_after.csv
c:\Users\z2918\Downloads\output\day04_project\data_quality_before.csv
c:\Users\z2918\Downloads\output\day04_project\ecommerce_customer_cleaned.csv
c:\Users\z2918\Downloads\output\day04_project\outlier_check_report.csv


## 项目复盘

请在提交前用不超过 200 字回答：

1. 本项目发现了哪些数据质量问题？
2. 你对缺失值、类别不一致、候选异常值分别采取了什么策略？
3. 为什么清洗后的数据可以作为第五天分析的输入？
4. 哪些处理规则仍需要业务人员确认？

1.发现的数据质量问题：数值字段存在缺失、分类字段同义标签不统一、存在完全重复行、部分指标存在 IQR 候选异常、少量违反业务逻辑的不合理数值。
2.处理策略：缺失值用全局中位数填充；同义类别做标准化合并；候选异常仅记录不盲目删除；重复行直接移除；业务违规值留存复核。
3.适配后续分析原因：数据缺失清零、分类口径统一、新增分层特征、全流程可追溯留日志，满足建模输入规范。
4.需业务确认的规则：IQR 检出的候选异常、业务逻辑违规极值，需要业务判断是否剔除或修正